# 01 — Data Exploration

This notebook explores the fox-detection audio dataset.

**Contents:**
1. Load `manifest.csv` and inspect class distribution.
2. Plot waveforms and spectrograms for 3 fox and 3 non-fox clips.
3. Plot duration histogram.
4. Plot average log-mel spectrogram per class.

> All file-loading cells are wrapped with `try/except` so the notebook stays
> runnable even when data files are not yet present.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

# ── Ensure project root is on sys.path ──────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.audio_utils import load_audio, normalise_audio, compute_log_mel_spectrogram

SR = 22050
MANIFEST_PATH = os.path.join(PROJECT_ROOT, "data", "manifest.csv")
print(f"Project root : {PROJECT_ROOT}")
print(f"Manifest path: {MANIFEST_PATH}")

## 1. Load manifest & class distribution

In [ ]:
try:
    df = pd.read_csv(MANIFEST_PATH)
    print(f"Manifest loaded: {len(df)} clips\n")
    display(df.head(10))
    print("\nClass distribution:")
    print(df["label"].value_counts())

    fig, ax = plt.subplots(figsize=(4, 3))
    df["label"].value_counts().plot.bar(ax=ax, color=["steelblue", "salmon"])
    ax.set_title("Class Distribution")
    ax.set_ylabel("Count")
    fig.tight_layout()
    plt.show()
except FileNotFoundError:
    print("⚠️  manifest.csv not found — run segmentation first.")
    df = pd.DataFrame(columns=["clip_path", "label", "file_id"])

## 2. Waveforms & spectrograms for 3 fox + 3 non-fox clips

In [ ]:
def _resolve(clip_path):
    """Resolve clip_path relative to manifest parent directory."""
    return os.path.join(os.path.dirname(MANIFEST_PATH), clip_path)


def plot_clip(clip_path, label, ax_wave, ax_spec):
    y, sr = load_audio(_resolve(clip_path), target_sr=SR)
    y = normalise_audio(y)
    t = np.arange(len(y)) / sr
    ax_wave.plot(t, y, linewidth=0.4, color="steelblue")
    ax_wave.set_title(f"{label}: {os.path.basename(clip_path)}", fontsize=9)
    ax_wave.set_xlabel("Time (s)")
    ax_wave.set_ylabel("Amp")

    spec = compute_log_mel_spectrogram(y, sr)
    ax_spec.imshow(spec, aspect="auto", origin="lower", cmap="magma")
    ax_spec.set_title("Log-mel spectrogram", fontsize=9)
    ax_spec.set_xlabel("Frame")
    ax_spec.set_ylabel("Mel bin")


N = 3  # samples per class

try:
    fox_rows = df[df["label"] == "fox"].head(N)
    nfx_rows = df[df["label"] == "nonfox"].head(N)

    if len(fox_rows) == 0 or len(nfx_rows) == 0:
        raise ValueError("Not enough clips in one of the classes.")

    total = len(fox_rows) + len(nfx_rows)
    fig, axes = plt.subplots(total, 2, figsize=(12, 2.5 * total))
    for i, (_, row) in enumerate(fox_rows.iterrows()):
        plot_clip(row["clip_path"], "fox", axes[i, 0], axes[i, 1])
    for i, (_, row) in enumerate(nfx_rows.iterrows()):
        plot_clip(row["clip_path"], "nonfox", axes[N + i, 0], axes[N + i, 1])
    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠️  Could not plot sample clips: {e}")

## 3. Duration histogram

In [ ]:
try:
    if len(df) == 0:
        raise FileNotFoundError("empty manifest")

    durations = []
    for _, row in df.iterrows():
        fp = _resolve(row["clip_path"])
        if os.path.isfile(fp):
            y, sr = load_audio(fp, target_sr=SR)
            durations.append(len(y) / sr)

    if durations:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.hist(durations, bins=30, edgecolor="white", color="steelblue")
        ax.axvline(np.mean(durations), color="red", ls="--",
                   label=f"Mean = {np.mean(durations):.2f} s")
        ax.set_xlabel("Duration (s)")
        ax.set_ylabel("Count")
        ax.set_title(f"Clip Duration Histogram  (n = {len(durations)})")
        ax.legend()
        fig.tight_layout()
        plt.show()
    else:
        print("⚠️  No audio files found on disk.")
except Exception as e:
    print(f"⚠️  Duration histogram skipped: {e}")

## 4. Average log-mel spectrogram per class

In [ ]:
try:
    if len(df) == 0:
        raise FileNotFoundError("empty manifest")

    avg_specs = {}
    for label in ["fox", "nonfox"]:
        subset = df[df["label"] == label]
        specs = []
        for _, row in subset.iterrows():
            fp = _resolve(row["clip_path"])
            if not os.path.isfile(fp):
                continue
            y, sr = load_audio(fp, target_sr=SR)
            y = normalise_audio(y)
            specs.append(compute_log_mel_spectrogram(y, sr))
        if specs:
            min_w = min(s.shape[1] for s in specs)
            avg_specs[label] = np.mean([s[:, :min_w] for s in specs], axis=0)

    if avg_specs:
        fig, axes = plt.subplots(1, len(avg_specs),
                                 figsize=(6 * len(avg_specs), 4))
        if not hasattr(axes, "__len__"):
            axes = [axes]
        for ax, (label, spec) in zip(axes, avg_specs.items()):
            im = ax.imshow(spec, aspect="auto", origin="lower", cmap="magma")
            ax.set_title(f"Average spectrogram — {label}")
            ax.set_xlabel("Frame")
            ax.set_ylabel("Mel bin")
            fig.colorbar(im, ax=ax, format="%+2.0f dB")
        fig.tight_layout()
        plt.show()
    else:
        print("⚠️  No spectrograms could be computed.")
except Exception as e:
    print(f"⚠️  Average spectrogram skipped: {e}")